# Capital Asset Pricing Model

Quis Question

Consider a market with d = 3 risky assets and 1 risk-free asset with the following parameters:
* u = [6%, 2%, 4%]
* V = [[8, -2, 4], [-2, 2, -2], [4, -2, 8]] * 10^-3
* rf = 1%

Note that the mean returns on the assets are percentage but the variance is not. You can compute the answers to these questions either by directly using the data above. 

In [1]:
import numpy as np
import math
from scipy.optimize import minimize
from numpy.linalg import inv, pinv

In [3]:
# load parameters
returns = np.array([6, 2, 4])/100
covariance = np.array([[8, -2, 4],[-2, 2, -2],[4, -2, 8]])/1000
risk_free_return = 1/100

covariance

array([[ 0.008, -0.002,  0.004],
       [-0.002,  0.002, -0.002],
       [ 0.004, -0.002,  0.008]])

In [12]:
volatility = np.diag(covariance)
excess_return = returns - risk_free_return

excess_return

array([0.05, 0.01, 0.03])

In [13]:
volatility

array([0.008, 0.002, 0.008])

## Q1: Compute the mean return on the portfolio x=1/3(1,1,1) consisting only of the risky assets

- weighted average of returns

In [8]:
x_porfolio = np.array([1, 1, 1])*1/3
x_return = np.dot(returns, x_porfolio)
# or
x_return = returns @ x_porfolio

print(f'Mean return (%): {round(x_return*100, 2)}')

Mean return (%): 4.0


## Q2: Compute the volatility of the return
- volatility is standard deviation

In [ ]:
x_variance = x_porfolio @ covariance @ x_porfolio
x_volatility = math.sqrt(x_variance)

print(f'Volatility (%): {round(x_volatility*100, 2)}')

Volatility (%): 4.47


## Q3: Compute the mean return on the minimum variance portfolio of just the risky assets. 
- The minimum variance portfolio is defined as the portfolio of risky assets that has the least volatility among all possible portfolios of just the risky assets

In [11]:
def calculate_portfolio_variance(x, V):
    return x @ V @ x # returns the portfolio covariance

constraints = (
    {
        'type': 'eq',
        'fun': lambda x:  np.sum(x)-1
    }
)

min_optimizer = minimize(
    calculate_portfolio_variance,
    x_porfolio,
    args = covariance,
    method = 'SLSQP', # Sequential Least Squares Programming
    constraints = constraints,
    tol = 1e-6
)

# minimum variance portfolio
min_var_portfolio = min_optimizer.x
min_var_portfolio_mean_return = returns @ min_var_portfolio

print(f'Minimum variance portfolio mean return (%): {round(min_var_portfolio_mean_return*100, 2)}')

Minimum variance portfolio mean return (%): 3.0


## Q4: Compute the mean return on the Sharpe optimal portfolio for this market.
- Recall that we can compute the Sharpe optimal portfolio by taking the risky positions corresponding to any risk aversion parameter and re-scaling them so that the components add up to 1

In [15]:
# sharpe ratio = (expected portfolio return - risk free rate)/ portfolio volatility
## 3 steps:
# 1. excess return
excess_return = returns - risk_free_return
# 2. compute unscaled portfolio
w = np.linalg.inv(covariance) @  excess_return
# 3. normalize portfolio
w = w/ np.sum(w)

sharpe_opt_portfolio_return = returns @ w

print(f'Sharpe optimal portfolio mean return (%): {round(sharpe_opt_portfolio_return*100, 2)}')

Sharpe optimal portfolio mean return (%): 3.42


## Q5: Compute the volatility of the Sharpe optimal portfolio for this market

In [17]:
sharpe_opt_portfolio_volatility = math.sqrt(w @ covariance @ w)
print(f'Sharpe optimal portfolio volatility (%): {round(sharpe_opt_portfolio_volatility*100, 2)}')

Sharpe optimal portfolio volatility (%): 2.84


## Q6: Using the results in the previous question, compute the slope of the capital market line

In [18]:
slope = (sharpe_opt_portfolio_return - risk_free_return)/ sharpe_opt_portfolio_volatility
print(f'Slope: {round(slope, 2)}')

Slope: 0.85
